# Project Kinetica — Edge Traffic Perception Model Training
**Course:** BCSE497J Project I (VIT Chennai)  
**Target Architecture:** YOLOv8 Nano Edge Model (GPU Accelerated)  
**Classes (7):** `car`, `bus`, `truck`, `motorcycle`, `auto_rickshaw`, `ambulance`, `police`  
**Google Drive Dataset:** `https://drive.google.com/file/d/1Hxr0KxgEZFwatjf50VtIZNigci1dfie9/view?usp=sharing`

---
### Instructions:
1. In Google Colab menu: **Runtime → Change runtime type → T4 GPU** (already set by default).
2. Click **Runtime → Run all** (or press `Ctrl + F9`).
3. The notebook will automatically download the 5.3 GB dataset from Google Drive via `gdown`, extract it, train for 35 epochs on the GPU (~25–35 mins), and automatically download `best.pt` to your computer when finished!

In [ ]:
# Step 1: Verify NVIDIA GPU Environment
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available: ", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Active GPU:     ", torch.cuda.get_device_name(0))
    print("VRAM Allocated: ", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")
else:
    raise RuntimeError("❌ GPU not detected! Please go to Runtime -> Change runtime type -> Select T4 GPU")

In [ ]:
# Step 2: Install Ultralytics and Gdown
!pip install -q ultralytics gdown

In [ ]:
# Step 3: Automatically Download Dataset from Google Drive & Unzip
import os, zipfile

GDRIVE_FILE_ID = "1Hxr0KxgEZFwatjf50VtIZNigci1dfie9"
DEST_ZIP = "/content/kinetica_dataset.zip"

# Download if not already present
if not os.path.exists(DEST_ZIP):
    print("Downloading 5.3 GB kinetica_dataset.zip directly from Google Drive...")
    !gdown --id {GDRIVE_FILE_ID} -O {DEST_ZIP}
else:
    print(f"{DEST_ZIP} already exists!")

# Extract dataset
print("Extracting kinetica_dataset.zip to /content/ ...")
with zipfile.ZipFile(DEST_ZIP, 'r') as zip_ref:
    zip_ref.extractall('/content')

print("\n✅ Extraction complete!")
print("Train images:", len(os.listdir('/content/kinetica_dataset/images/train')))
print("Val images:  ", len(os.listdir('/content/kinetica_dataset/images/val')))

In [ ]:
# Step 4: Verify Dataset Structure & YAML Configuration
import os

yaml_path = '/content/kinetica_dataset/kinetica.yaml'

if os.path.exists(yaml_path):
    with open(yaml_path, 'r') as f:
        print("--- kinetica.yaml ---")
        print(f.read())
else:
    print("Writing kinetica.yaml fallback...")
    with open(yaml_path, 'w') as f:
        f.write("""path: /content/kinetica_dataset
train: images/train
val: images/val

nc: 7
names:
  0: car
  1: bus
  2: truck
  3: motorcycle
  4: auto_rickshaw
  5: ambulance
  6: police
""")
    print("kinetica.yaml written successfully.")

In [ ]:
# Step 5: Fine-Tune Custom YOLOv8n on GPU
from ultralytics import YOLO

# 1. Load pretrained base YOLOv8 nano model
model = YOLO('yolov8n.pt')

# 2. Train on GPU (device=0 forces NVIDIA CUDA acceleration)
# Approx. 25-35 minutes on Colab T4 GPU for 35 epochs
results = model.train(
    data='/content/kinetica_dataset/kinetica.yaml',
    epochs=35,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    project='/content/runs/train',
    name='kinetica_yolov8n',
    patience=10,
    save=True,
    verbose=True
)

print("\n Training Complete!")

In [ ]:
# Step 6: Validate Metrics & Export to ONNX
metrics = model.val()
print("\n--- MODEL VALIDATION PERFORMANCE ---")
print(f"Validation mAP@0.5:      {metrics.box.map50:.4f}")
print(f"Validation mAP@0.5:0.95: {metrics.box.map:.4f}")

# Export to ONNX for NVIDIA Jetson / Edge deployment
onnx_path = model.export(format='onnx')
print(f"Exported ONNX model to: {onnx_path}")

In [ ]:
# Step 7: Automatically Download best.pt to Your Computer
from google.colab import files
import os

best_weights = '/content/runs/train/kinetica_yolov8n/weights/best.pt'

if os.path.exists(best_weights):
    print(f"Found best weights at {best_weights} ({os.path.getsize(best_weights) / 1e6:.1f} MB)")
    print("Triggering browser download to your local PC...")
    files.download(best_weights)
else:
    print("Error: Could not find best.pt!")